In [30]:
!pip install -q transformers datasets accelerate

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import time

In [31]:
import gc
torch.cuda.empty_cache()
gc.collect()

4667

In [32]:
model_name = "EleutherAI/pythia-1b"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
model.eval()

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 2048)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-15): 16 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=2048, out_features=6144, bias=True)
          (dense): Linear(in_features=2048, out_features=2048, bias=True)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=2048, out_features=8192, bias=True)
          (dense_4h_to_h): Linear(in_features=8192, out_features=2048, bias=True)
          (act): GELUActivation()
        )
      )
    )
    (final_layer_norm): LayerNorm((2048,), eps=1e-05, 

In [33]:
dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="test[:1%]")

# remove empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids"])

In [34]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [35]:
def compute_loss(logits, labels):
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]

    return F.cross_entropy(
        shift_logits.reshape(-1, shift_logits.size(-1)),
        shift_labels.reshape(-1),
        ignore_index=tokenizer.pad_token_id
    )

In [36]:
print("Total layers:", len(model.gpt_neox.layers))

Total layers: 16


In [37]:
class SkipLayer(torch.nn.Module):
    def __init__(self, layer, layer_id, skip_layers_ref):
        super().__init__()
        self.layer = layer
        self.layer_id = layer_id
        self.skip_layers_ref = skip_layers_ref  # reference, not copy

    def forward(self, hidden_states, *args, **kwargs):
        if self.layer_id in self.skip_layers_ref:
            return (hidden_states, None)
        return self.layer(hidden_states, *args, **kwargs)

In [38]:
skip_layers_ref = set()

def apply_skip(model):
    for i, layer in enumerate(model.gpt_neox.layers):
        model.gpt_neox.layers[i] = SkipLayer(layer, i, skip_layers_ref)

In [39]:
model = AutoModelForCausalLM.from_pretrained(model_name).half().cuda()
model.eval()
apply_skip(model)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [40]:
total_layers = len(model.gpt_neox.layers)

configs = {
    "full": set(),
    "skip_25": set(range(int(0.75 * total_layers), total_layers)),
    "skip_50": set(range(int(0.5 * total_layers), total_layers)),
}

In [41]:
results = {}

for name, skip_layers in configs.items():
    print(f"Running: {name}")

    # update dynamic skip set
    skip_layers_ref.clear()
    skip_layers_ref.update(skip_layers)

    total_loss = 0
    count = 0

    start = time.time()

    for batch in dataset:
        input_ids = batch["input_ids"].unsqueeze(0).cuda()

        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits

        loss = compute_loss(logits, input_ids)
        total_loss += loss.item()
        count += 1

    elapsed = time.time() - start

    results[name] = {
        "loss": total_loss / count,
        "time_sec": elapsed
    }

Running: full
Running: skip_25
Running: skip_50


In [42]:
for k, v in results.items():
    print(f"{k}: loss={v['loss']:.4f}, time={v['time_sec']:.2f}s")

full: loss=5.5540, time=0.41s
skip_25: loss=7.3484, time=0.27s
skip_50: loss=8.7852, time=0.20s


In [43]:
import torch.nn.functional as F

def compute_entropy(logits):
    logits = logits.float()

    log_probs = torch.log_softmax(logits, dim=-1)
    probs = torch.exp(log_probs)

    entropy = -(probs * log_probs).sum(dim=-1)
    return entropy.mean()

In [45]:
entropy = compute_entropy(logits)  # (batch, seq)
token_entropy = entropy.mean(dim=0)
if torch.isnan(entropy):
    print("NaN detected!")
    entropy = torch.tensor(10.0).cuda()  # fallback

In [46]:
class EntropySkipLayer(torch.nn.Module):
    def __init__(self, layer, layer_id, config):
        super().__init__()
        self.layer = layer
        self.layer_id = layer_id
        self.config = config

    def forward(self, hidden_states, *args, **kwargs):
        if not self.config["enable_skip"]:
            return self.layer(hidden_states, *args, **kwargs)

        if self.layer_id < self.config["min_layers"]:
            return self.layer(hidden_states, *args, **kwargs)

        if self.config["current_entropy"] < self.config["threshold"]:
            self.config["skipped"] += 1
            return (hidden_states, None)

        return self.layer(hidden_states, *args, **kwargs)

In [48]:
config = {
    "threshold": 2.5,
    "min_layers": 6,
    "current_entropy": 0.0,
    "skipped": 0,
    "enable_skip": False   # NEW
}

In [49]:
def apply_entropy_skip(model, config):
    for i, layer in enumerate(model.gpt_neox.layers):
        model.gpt_neox.layers[i] = EntropySkipLayer(layer, i, config)

In [50]:
model = AutoModelForCausalLM.from_pretrained(model_name).half().cuda()
model.eval()

apply_entropy_skip(model, config)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [51]:
results = {}

thresholds = [3.0, 3.5, 4.0, 4.5]
for t in thresholds:
    print(f"\nRunning threshold: {t}")

    config["threshold"] = t

    total_loss = 0
    total_skips = 0
    count = 0

    start = time.time()

    for batch in dataset:
        input_ids = batch["input_ids"].unsqueeze(0).cuda()

    with torch.no_grad():
        # PASS 1: no skipping
        config["enable_skip"] = False
        outputs = model(input_ids)
        logits = outputs.logits

        entropy = compute_entropy(logits)
        config["current_entropy"] = entropy.item()

        # PASS 2: with skipping
        config["enable_skip"] = True
        config["skipped"] = 0

        outputs = model(input_ids)
        logits = outputs.logits

        loss = compute_loss(logits, input_ids)

        total_loss += loss.item()
        total_skips += config["skipped"]
        count += 1

    elapsed = time.time() - start

    results[t] = {
        "loss": total_loss / count,
        "time_sec": elapsed,
        "avg_skips": total_skips / count
    }


Running threshold: 3.0

Running threshold: 3.5

Running threshold: 4.0

Running threshold: 4.5


In [52]:
for k, v in results.items():
    print(f"threshold={k}: loss={v['loss']:.4f}, time={v['time_sec']:.2f}s, skips={v['avg_skips']:.2f}")

threshold=3.0: loss=3.0898, time=0.05s, skips=0.00
threshold=3.5: loss=3.0898, time=0.05s, skips=0.00
threshold=4.0: loss=8.0391, time=0.04s, skips=10.00
threshold=4.5: loss=8.0391, time=0.04s, skips=10.00


In [53]:
print(entropy.print("entropy:", entropy.item())
print("threshold:", config["threshold"])())

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1473687546.py, line 1)

V3

In [54]:
def compute_loss(logits, labels):
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]

    return F.cross_entropy(
        shift_logits.reshape(-1, shift_logits.size(-1)),
        shift_labels.reshape(-1),
        ignore_index=tokenizer.pad_token_id
    )

In [55]:
for i, layer in enumerate(model.gpt_neox.layers):
    layer.layer_id = i

In [56]:
features, targets = [], []

def hook_fn(module, input, output):
    layer_id = module.layer_id  # we’ll attach this

    h_in = input[0].detach().float()
    h_out = output[0].detach().float()

    delta = (h_out - h_in).abs().mean(dim=[1,2])
    delta = torch.clamp(delta, 0, 10)

    feat = torch.stack([
        h_in.norm(dim=-1).mean(dim=1),
        h_in.abs().mean(dim=[1,2]),
        torch.full_like(delta, layer_id / 24.0)  # 🔥 layer info
    ], dim=1)

    features.append(feat.cpu())
    targets.append(delta.cpu())

In [57]:
hooks = []
for layer in model.gpt_neox.layers:
    hooks.append(layer.register_forward_hook(hook_fn))

for i, batch in enumerate(dataset):
    if i > 50: break
    input_ids = batch["input_ids"].unsqueeze(0).cuda()
    with torch.no_grad():
        _ = model(input_ids)

for h in hooks:
    h.remove()

In [58]:
X = torch.cat(features)
Y = torch.cat(targets)

# remove bad values
mask = torch.isfinite(X).all(dim=1) & torch.isfinite(Y)
X = X[mask]
Y = Y[mask]

# normalize
X_mean, X_std = X.mean(0), X.std(0) + 1e-6
Y_mean, Y_std = Y.mean(), Y.std() + 1e-6

X = (X - X_mean) / X_std
Y = (Y - Y_mean) / Y_std

In [59]:
dataset_voc = torch.utils.data.TensorDataset(X, Y)
dataloader = torch.utils.data.DataLoader(dataset_voc, batch_size=32, shuffle=True)

NameError: name 'x' is not defined

In [66]:
import torch
import torch.nn as nn

class VoCModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

voc_model = VoCModel().cuda().float()

In [67]:
import torch.nn.functional as F

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(voc_model.parameters(), lr=1e-4)

for epoch in range(5):
    total_loss = 0

    for x, y in dataloader:
        x = x.float().cuda()
        y = y.float().cuda().unsqueeze(1)

        pred = voc_model(x)

        loss = criterion(pred, y)

        if torch.isnan(loss):
            print("NaN detected, skipping batch")
            continue

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(voc_model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    print(f"epoch {epoch} loss {total_loss:.4f}")

epoch 0 loss 9.3526
epoch 1 loss 9.2002
epoch 2 loss 9.1019
epoch 3 loss 8.8860
epoch 4 loss 8.7217


In [112]:
import torch.nn as nn

class Router(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

In [104]:
router = Router().cuda().float()

In [105]:
class VoCSkipLayer(torch.nn.Module):
    def __init__(self, layer, layer_id, config, router):
        super().__init__()
        self.layer = layer
        self.layer_id = layer_id
        self.config = config
        self.router = router

        for name, module in layer.named_children():
            setattr(self, name, module)

    def forward(self, hidden_states, *args, **kwargs):
        prev = self.config.get("prev_hidden", None)
        if prev is None:
            prev = hidden_states

        # ---- features ----
        delta = (hidden_states - prev).norm(dim=-1).mean(dim=1)
        var = hidden_states.var(dim=1).mean(dim=1)
        ratio = hidden_states.norm(dim=[1,2]) / (prev.norm(dim=[1,2]) + 1e-6)

        # ---- layer context ----
        layer_pos = torch.full(
            (hidden_states.size(0),),
            self.layer_id / self.config["num_layers"],
            device=hidden_states.device
        )

        feat = torch.stack([delta, var, ratio, layer_pos], dim=1).float()

        self.config["prev_hidden"] = hidden_states.detach()

        # =========================
        # DATA COLLECTION
        # =========================
        if self.config.get("collect_data", False):
            with torch.no_grad():
                full_out = self.layer(hidden_states, *args, **kwargs)
                full_h = full_out[0] if isinstance(full_out, tuple) else full_out

            skip_h = hidden_states

            # better importance (per-token avg)
            importance = (full_h - skip_h).norm(dim=-1).mean().item()

            self.config["records"].append({
                "feat": feat.detach().cpu(),
                "label": importance
            })

            return full_out

        # =========================
        # INFERENCE
        # =========================
        score = self.router(feat).mean().item()

        if score < self.config["threshold"]:
            self.config["skipped"] += 1

            if kwargs.get("use_cache", False):
                return (hidden_states, None, None)
            else:
                return (hidden_states, None)

        return self.layer(hidden_states, *args, **kwargs)

In [106]:
config = {
    "threshold": 0.0,
    "skipped": 0,
    "prev_hidden": None,
    "collect_data": False,
    "records": [],
    "num_layers": len(model.gpt_neox.layers)
}

for i, layer in enumerate(model.gpt_neox.layers):
    model.gpt_neox.layers[i] = VoCSkipLayer(layer, i, config, router)

In [107]:
config["collect_data"] = True
config["records"] = []

for batch in dataset:
    input_ids = batch["input_ids"].unsqueeze(0).cuda()

    with torch.no_grad():
        config["prev_hidden"] = None
        _ = model(input_ids)

records = config["records"]
print("collected:", len(records))

collected: 336


In [108]:
router.train()
opt = torch.optim.Adam(router.parameters(), lr=1e-3)

for epoch in range(3):
    for r in records:
        x = r["feat"].cuda()
        y = torch.tensor([r["label"]]).cuda()

        pred = router(x).squeeze()
        loss = (pred - y) ** 2

        opt.zero_grad()
        loss.backward()
        opt.step()

In [109]:
config["collect_data"] = False

In [113]:
results = {}
thresholds = [-1.0, -0.5, 0.0, 0.5, 1.0]

for t in thresholds:
    config["threshold"] = t

    total_loss, total_skips, count = 0, 0, 0
    start = time.time()

    for batch in dataset:
        input_ids = batch["input_ids"].unsqueeze(0).cuda()

        with torch.no_grad():
            config["skipped"] = 0
            config["prev_hidden"] = None
            outputs = model(input_ids)
            logits = outputs.logits

        loss = compute_loss(logits, input_ids)

        total_loss += loss.item()
        total_skips += config["skipped"]
        count += 1

    elapsed = time.time() - start

    results[t] = {
        "loss": total_loss / count,
        "time_sec": elapsed,
        "avg_skips": total_skips / count
    }

for k, v in results.items():
    print(f"t={k}: loss={v['loss']:.4f}, time={v['time_sec']:.2f}s, skips={v['avg_skips']:.2f}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x4 and 3x32)